# Cervical Cancer Stage Classification on Kaggle

This notebook trains the cervical cancer classifier on Kaggle and writes checkpoints + metrics to `/kaggle/working/Checkpoints`.

**This version embeds a fixed `train.py`** (imbalance double-counting removed, gentler domain-appropriate augmentation, EMA-smoothed checkpointing, warmup+cosine LR, gradient accumulation). It overwrites whatever `train.py` is in the cloned repo, so it runs correctly even if the GitHub repo hasn't been updated yet.

## Kaggle Run Guide

1. Open this notebook in Kaggle, attach a GPU accelerator (Settings → Accelerator → GPU T4 x2 or P100).
2. Attach the Herlev dataset as a Kaggle Input (or set `DATA_DIR` manually in Cell 3 if your dataset path differs).
3. Turn Internet **on** in notebook settings (needed to `git clone` the repo and pull pretrained backbone weights).
4. Run all cells top to bottom.
5. Checkpoints land in `/kaggle/working/Checkpoints/`: `best_model.pt`, `last_model.pt`, `history.json`, `metrics.json`.
6. Use the Kaggle "Save Version" → "Save & Run All" to persist `/kaggle/working/Checkpoints` as notebook output.

In [ ]:
from __future__ import annotations

import base64
import os
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_NAME = 'Cervical-Cancer-Classifier'

# On notebook re-runs the kernel's cwd can be left INSIDE the repo folder from a
# previous run (we os.chdir(REPO_ROOT) below). If we then rm -rf that same folder
# before removing it we'd be deleting our own cwd out from under the process,
# which breaks os.getcwd() and makes `git clone` fail with:
#   "fatal: Unable to read current working directory: No such file or directory"
# So always chdir somewhere safe FIRST, before any cleanup/removal happens.
SAFE_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.home()
os.chdir(SAFE_ROOT)

# The fixed train.py handles its own runtime dependencies and installs timm if needed.

def find_backend_dir() -> Path | None:
    search_roots = [Path('/kaggle/working'), Path('/kaggle/input'), Path.cwd()]
    direct_candidates = [
        Path('/kaggle/working/backend'),
        Path('/kaggle/input/backend'),
        Path.cwd() / 'backend',
        Path('/kaggle/working') / REPO_NAME / 'backend',
        Path('/kaggle/input') / REPO_NAME / 'backend',
    ]
    for candidate in direct_candidates:
        if (candidate / 'train.py').exists():
            return candidate
    for root in search_roots:
        if not root.exists():
            continue
        for match in root.rglob('train.py'):
            if match.name == 'train.py' and match.parent.name == 'backend':
                return match.parent
    return None


def clone_repo_if_needed() -> Path:
    work_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    repo_dir = work_root / REPO_NAME
    backend_dir = repo_dir / 'backend'
    if (backend_dir / 'train.py').exists():
        return backend_dir

    if repo_dir.exists():
        print(f'Removing incomplete repo folder: {repo_dir}')
        os.chdir(SAFE_ROOT)  # never rm -rf a directory we might be sitting inside of
        subprocess.run(['rm', '-rf', str(repo_dir)], check=False)

    print(f'Cloning repository from {REPO_URL}')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    if not (backend_dir / 'train.py').exists():
        raise FileNotFoundError(f'Cloned repo but could not find backend/train.py in {backend_dir}')
    return backend_dir


BACKEND_DIR = find_backend_dir()
if BACKEND_DIR is None:
    BACKEND_DIR = clone_repo_if_needed()

REPO_ROOT = BACKEND_DIR.parent
os.chdir(REPO_ROOT)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# ── Write the fixed train.py (base64-embedded so it runs correctly on Kaggle
# regardless of what is currently pushed to GitHub) ─────────────────────────
TRAIN_PY_B64 = '...'

train_script_path = BACKEND_DIR / 'train.py'
train_script_path.write_bytes(base64.b64decode(TRAIN_PY_B64))
train_script_text = train_script_path.read_text()
train_script_text = re.sub(r'^from\s+models\.[A-Za-z0-9_]+\s+import\s+build_model$', 'from models.hybrid_model import build_model', train_script_text, flags=re.MULTILINE)
train_script_text = train_script_text.replace('models.cnn_model', 'models.hybrid_model')
train_script_path.write_text(train_script_text)
if 'models.cnn_model' in train_script_path.read_text():
    raise RuntimeError('train.py patch did not remove the legacy cnn_model import')
print('Wrote fixed train.py ->', train_script_path, f'({train_script_path.stat().st_size} bytes)')

print('REPO_ROOT:', REPO_ROOT)
print('BACKEND_DIR:', BACKEND_DIR)
print('Python:', sys.executable)

In [ ]:
CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']


def _class_count(base: Path) -> int:
    try:
        return sum(1 for name in CLASS_NAMES if (base / name).is_dir())
    except Exception:
        return 0


def _has_class_folders(base: Path) -> bool:
    return base.exists() and base.is_dir() and _class_count(base) == len(CLASS_NAMES)


def _looks_like_dataset_root(base: Path) -> bool:
    if not base.exists() or not base.is_dir():
        return False
    if _has_class_folders(base):
        return True
    for split_name in ('train', 'val', 'test'):
        split_dir = base / split_name
        if _has_class_folders(split_dir):
            return True
    return False


def find_data_dir() -> Path | None:
    raw_candidates = [
        os.environ.get('DATA_DIR', ''),
        '/kaggle/input/datasets/shubhrawat132/herlevdataset',
        '/kaggle/input/Herlev Dataset',
        '/kaggle/input/herlev-dataset',
        '/kaggle/input/herlevdataset',
        '/kaggle/input/cervical-cancer-stage-classification',
        '/kaggle/input/cervical-cancer-dataset',
        str(REPO_ROOT / 'Herlev Dataset'),
        str(REPO_ROOT / 'data'),
    ]
    for raw in raw_candidates:
        if not raw:
            continue
        candidate = Path(raw)
        if _looks_like_dataset_root(candidate):
            return candidate

    for root in [Path('/kaggle/input')]:
        if not root.exists():
            continue
        try:
            for candidate in root.iterdir():
                if candidate.is_dir() and _looks_like_dataset_root(candidate):
                    return candidate
                for sub in candidate.rglob('*'):
                    if sub.is_dir() and _looks_like_dataset_root(sub):
                        return sub
        except Exception:
            pass

    return None


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working/Checkpoints') if Path('/kaggle/working').exists() else (REPO_ROOT / 'backend' / 'Checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    raise FileNotFoundError('Could not find the dataset. Set DATA_DIR to your Kaggle input folder and rerun this cell, e.g. os.environ["DATA_DIR"] = "/kaggle/input/your-dataset-slug"')

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Class folders found at DATA_DIR:', {name: (DATA_DIR / name).exists() for name in CLASS_NAMES})

In [ ]:
train_script = BACKEND_DIR / 'train.py'

command = [
    sys.executable, '-u', str(train_script),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '90',
    '--phase1-epochs', '24',
    '--phase2-epochs', '26',
    '--batch-size', '16',
    '--accum-steps', '2',
    '--img-size', '224',
    '--backbone', 'tf_efficientnetv2_s',
    '--dropout', '0.35',
    '--lr-head', '1.5e-4',
    '--lr-backbone', '2e-5',
    '--focal-gamma', '1.5',
    '--label-smoothing', '0.05',
    '--mixup-alpha', '0.20',
    '--mixup-prob', '0.30',
    '--ema-decay', '0.999',
    '--patience', '20',
    '--val-frac', '0.20',
    '--num-workers', str(min(4, os.cpu_count() or 2)),
    # NOTE: imbalance is handled by the WeightedRandomSampler alone by default.
    # Uncomment the next line to also A/B test class-weighted loss on top of it:
    # '--use-loss-weighting',
]

print('Running:')
print(' '.join(command))
print()

result = subprocess.run(command, check=False)
if result.returncode != 0:
    raise RuntimeError(f'train.py exited with code {result.returncode}. Scroll up for the traceback.')

In [ ]:
print('Training artifacts written to:', OUTPUT_DIR)
artifacts = sorted(OUTPUT_DIR.glob('*'))
if not artifacts:
    print('No artifacts found yet. Run the training cell first.')
else:
    print('Training artifacts:')
    for artifact in artifacts:
        print('-', artifact.name)

for name in ['best_model.pt', 'last_model.pt', 'history.json', 'metrics.json']:
    path = OUTPUT_DIR / name
    print(f'{name}:', path.exists())

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    import json as _json
    m = _json.loads(metrics_path.read_text())['final_metrics']
    print('\nFinal validation metrics:')
    for key in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'loss']:
        print(f'  {key:>18}: {m.get(key)}')